# User Goal Classification

⚠️ **SYNTHETIC DATA** — All training data is procedurally generated. Results do NOT reflect real-world user performance.

Compares LogisticRegression, RandomForest, GradientBoosting on the user wellness-goal classification task.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve().parent
sys.path.insert(0, str(ROOT))

In [ ]:
# Generate and inspect synthetic data
from ml.models.user_goal_classifier import generate_synthetic_profiles, GOAL_CLASSES
import pandas as pd

df = generate_synthetic_profiles(n_samples=2000)
print(f'Shape: {df.shape}')
print(f'\nGoal distribution:')
print(df['goal_label'].value_counts())
print(f'\nNote: _synthetic column = {df["_synthetic"].unique()}')

In [ ]:
# Feature distributions
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
feature_cols = ['age','bmi','activity_level','yoga_experience','flexibility_score',
                'strength_score','sleep_hours','session_duration_min','fitness_experience']
for ax, col in zip(axes.flat, feature_cols):
    df[col].hist(ax=ax, bins=20, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Synthetic Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train and compare models
# python -m ml.training.train_user_goal
print('Run: python -m ml.training.train_user_goal')

In [ ]:
# Load results (after training)
import json
metrics_path = ROOT / 'models' / 'metrics' / 'user_goal_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    for model_name, results in m.items():
        test = results['test']
        print(f"\n{model_name}")
        print(f"  Accuracy  : {test['accuracy']}  [data={test['data_source']}]")
        print(f"  F1 Score  : {test['f1_score']}")
        cv = results['cv']
        print(f"  CV Acc    : {cv['accuracy']['mean']} ± {cv['accuracy']['std']}")
else:
    print('Train first')

In [ ]:
# Confusion matrix for best model
import json, numpy as np, matplotlib.pyplot as plt
metrics_path = ROOT / 'models' / 'metrics' / 'user_goal_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    # Pick best by test accuracy
    best = max(m.items(), key=lambda x: x[1]['test']['accuracy'])
    name, data = best
    cm = np.array(data['test']['confusion_matrix'])
    from ml.models.user_goal_classifier import GOAL_CLASSES
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, cmap='Greens')
    plt.colorbar()
    plt.xticks(range(len(GOAL_CLASSES)), GOAL_CLASSES, rotation=45, ha='right')
    plt.yticks(range(len(GOAL_CLASSES)), GOAL_CLASSES)
    plt.title(f'{name} Confusion Matrix  [SYNTHETIC DATA]')
    plt.tight_layout()
    plt.show()
else:
    print('Train first')

## Limitations
- All data is SYNTHETIC with domain rules
- No real user-goal ground-truth dataset was available
- These metrics are illustrative only and must NOT be quoted as real-world performance